# 🧪 Lab 2 — MLP com Keras/TensorFlow (GABARITO)

**Uso do professor:** Resolução completa das 10 células.

**Baseline a bater:** XGBoost RMSE ~ 1.5%.

### Célula 1 — Importar bibliotecas

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

print(f"TensorFlow version: {tf.__version__}")

### Célula 2 — Carregar dados

In [ ]:
URL = "https://raw.githubusercontent.com/LuisGSVasconcelos/IA_EngQuimica/main/dados/aula10/reator_rendimento.csv"
df = pd.read_csv(URL, parse_dates=['timestamp'])
df.set_index('timestamp', inplace=True)

print(df.info())
print()
print(df.describe().round(2))
print()
print("Correlações com rendimento:")
print(df.corr(numeric_only=True)['rendimento_pct'].sort_values(ascending=False).round(3))

**Anotação:** `T_reator_C` e `conc_alimentacao` correlacionam com o rendimento. O rendimento tem um pico em T ótimo (relação não-linear).

### Célula 3 — Features + Split + Escalonamento

In [ ]:
df['T_lag1'] = df['T_reator_C'].shift(1)
df['T_lag2'] = df['T_reator_C'].shift(2)
df['T_ma3'] = df['T_reator_C'].rolling(3).mean()
df = df.dropna()

feat = ['T_reator_C', 'vazao_L_min', 'pressao_bar', 'conc_alimentacao_mol_L', 'T_lag1', 'T_lag2', 'T_ma3']
X = df[feat]
y = df['rendimento_pct']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print(f"Treino: {X_train.shape[0]} | Teste: {X_test.shape[0]} | Features: {X.shape[1]}")

### Célula 4 — Baseline XGBoost

In [ ]:
xgb = XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=42, verbosity=0)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)
print(f"XGBoost baseline: RMSE={rmse_xgb:.3f}%  R²={r2_xgb:.3f}")

### Célula 5 — Definir arquitetura

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])
model.summary()

**Justificativa:** Dense(64) captura não-linearidades; Dropout(0.2) reduz overfitting; Dense(32) compressão do sinal; Dense(1) = saída de regressão (linear).

### Célula 6 — Compilar + Callbacks

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss='mse', metrics=['mae'])
callbacks = [
    keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5)
]
print("Compilado com Adam(0.001) + EarlyStopping(15) + ReduceLROnPlateau")

### Célula 7 — Treinar + Loss Curve

In [ ]:
history = model.fit(X_train_s, y_train, epochs=200, validation_split=0.1,
                    callbacks=callbacks, batch_size=32, verbose=0)

plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Treino')
plt.plot(history.history['val_loss'], label='Validação')
plt.xlabel('Época'); plt.ylabel('Loss (MSE)')
plt.legend(); plt.grid(alpha=0.3); plt.title('Loss Curve — MLP Keras')
plt.show()

print(f"Épocas treinadas: {len(history.history['loss'])}")

**Anotação:** O early stopping interrompeu o treino entre ~40-80 épocas. O loss de validação converge e estabiliza.

### Célula 8 — Avaliar no teste

In [ ]:
test_loss, test_mae = model.evaluate(X_test_s, y_test, verbose=0)
y_pred_mlp = model.predict(X_test_s, verbose=0).ravel()
rmse_mlp = np.sqrt(mean_squared_error(y_test, y_pred_mlp))
r2_mlp = r2_score(y_test, y_pred_mlp)
print(f"MLP Keras:    RMSE={rmse_mlp:.3f}%  R²={r2_mlp:.3f}  MAE={test_mae:.3f}")

### Célula 9 — Comparar com baseline

In [ ]:
import pandas as pd
tabela = pd.DataFrame({
    'Modelo': ['XGBoost (baseline)', 'MLP Keras'],
    'RMSE (%)': [rmse_xgb, rmse_mlp],
    'R²': [r2_xgb, r2_mlp]
}).round(3)
print(tabela.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(y_test, y_pred_mlp, alpha=0.3, s=8)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0].set_title(f'MLP — RMSE: {rmse_mlp:.3f}')
axes[0].set_xlabel('Real'); axes[0].set_ylabel('Predito')
axes[1].scatter(y_test, y_pred_xgb, alpha=0.3, s=8, color='green')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[1].set_title(f'XGBoost — RMSE: {rmse_xgb:.3f}')
axes[1].set_xlabel('Real'); axes[1].set_ylabel('Predito')
plt.tight_layout()
plt.show()

### Célula 10 — Salvar modelo + Conclusão

In [ ]:
model.save('soft_sensor_reator.keras')
print("Modelo salvo: soft_sensor_reator.keras")

> **Conclusão:**
>
> **1. Atingiu/bateu o baseline?** A MLP Keras atinge RMSE ≈ 1.4-1.5%, igualando ou superando ligeiramente o XGBoost (~1.5%). O ganho é pequeno, como esperado para dados tabulares — o XGBoost é muito competitivo.
>
> **2. Épocas até early stopping:** ~40-80 épocas (com `patience=15`), variando conforme a arquitetura. O `ReduceLROnPlateau` ajudou a estabilizar a convergência.
>
> **3. Pronto para implantar?** Sim. O modelo salvo (`soft_sensor_reator.keras`) pode ser carregado e usado em produção. O valor do Keras está na flexibilidade (camadas customizadas, callbacks avançados) e na possibilidade de evoluir para arquiteturas mais complexas (LSTM, CNN).